# Step 1 — Model A: QLoRA fine-tune of MedGemma 1.5 4B (thin notebook: it only calls scripts)
**Settings:** Accelerator = GPU T4 x2, Internet = on. **Add-ons → Secrets:** tick `HF_TOKEN`.
**Inputs to attach:**
- `raddar/chest-xrays-indiana-university`
- your private `rexgradient-160k-test-images` (Step 0) and `rexgradient-160k-train-subset`
  (made once by the Kaggle job in `kaggle/fetch_rexgradient_train/`, then saved as a **Private** dataset)
- the **output of your finished Step 0 notebook** (holds the zero-shot predictions Model A is compared with)

Step 0 must have passed its acceptance check first. Training uses one T4 (Unsloth's open-source trainer is
single-GPU); Gemma 3 cannot train in plain fp16, so Unsloth switches to its float32-safe mode by itself.

In [ ]:
!git clone -q --depth 1 https://github.com/Vinith-44/Radioreport /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements-train.txt

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["RUN_NOTE"] = "step1 kaggle"   # free-text note logged with every run

## 0. Restore the Step 0 results (and a previous Step 1 session, to resume)
`/kaggle/working` is wiped between sessions. Point these at the attached outputs (look under `/kaggle/input`).

In [ ]:
STEP0 = None  # e.g. "/kaggle/input/step0-kaggle/project"  - REQUIRED for scoring (zero-shot baseline)
PREV1 = None  # e.g. "/kaggle/input/step1-kaggle/project"  - only to resume training / generation
os.makedirs("/kaggle/working/project", exist_ok=True)
if STEP0:
    !cp -rn {STEP0}/outputs {STEP0}/data /kaggle/working/project/
if PREV1:
    !cp -rn {PREV1}/outputs /kaggle/working/project/
!ls /kaggle/working/project/outputs/runs 2>/dev/null

## 1. Rebuild the parquet files so they include the train-subset images
The test split and the fixed test subset must come out IDENTICAL to Step 0 (otherwise the comparison is void):
the cell checks this.

In [ ]:
import json
ids_p = "/kaggle/working/project/data/step0_eval_ids.json"
before = json.load(open(ids_p)) if os.path.exists(ids_p) else None
!python scripts/step0_prepare_data.py --config configs/step1.yaml
after = json.load(open(ids_p))
assert before is None or before == after, "test eval ids changed - the Step 0 baseline is no longer comparable"
print("eval ids unchanged" if before else "WARNING: no Step 0 eval ids to compare with (attach STEP0)")
!python -c "import pandas as pd; d=pd.read_parquet('/kaggle/working/project/data/train.parquet'); print(d.groupby('dataset').size())"
!python -m pytest tests -q

## 2. Smoke run (~50 training studies, 6 steps) — check the sanity-check printout and outputs first

In [ ]:
!python scripts/step0_prepare_data.py --config configs/step1.yaml --smoke
!python scripts/step0_zeroshot_generate.py --model medgemma-1.5 --run-name zeroshot --smoke
!python scripts/step1_train_qlora.py --run-name a1 --smoke
!python scripts/step1_generate.py --run-name a1 --smoke
!python scripts/step1_score.py --run-name a1 --baseline-run zeroshot --smoke

## 3. Full training (resumable)
Prints s/step and the number of steps; it saves a checkpoint every 100 steps and stops cleanly after
`train.max_hours` (11 h). If it says NOT FINISHED: Save Version, attach that output as `PREV1` in a new
session, run sections 0-1 again and re-run this cell - it continues from the last checkpoint.

In [ ]:
!python scripts/step1_train_qlora.py --run-name a1

## 4. Test predictions + scoring vs zero-shot MedGemma 1.5 (same studies, paired test)

In [ ]:
!python scripts/step1_generate.py --run-name a1
!python scripts/step1_score.py --run-name a1 --baseline-run zeroshot
!cat /kaggle/working/project/outputs/RESULTS.md